In [1]:
"""
Coupled Pediatric Growth Forecasting via a Latent Neural ODE
=========================================================================
Runs on a cohort file with the columns of ``pediatric_cohort_coupled.csv``
(Child_ID, Age_Months, *_true, observed values, z_W / z_L / z_C, Observed).

"""

import os
import sys
import math
import subprocess
import argparse
import warnings

# -- Environment (Colab-aware) -------------------------------------------------
try:
    import google.colab                       # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

try:
    import torchdiffeq                         # noqa: F401
except ImportError:
    print("  Installing torchdiffeq ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchdiffeq"],
                   check=False)

import numpy as np
import pandas as pd
import matplotlib
if not IN_COLAB:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator, make_interp_spline
from scipy.optimize import minimize_scalar
from scipy.stats import norm

import torch
import torch.nn as nn
import torch.optim as optim
from torchdiffeq import odeint


def resolve_csv(cfg):
    """In Colab, ALWAYS prompt an upload (so you can see what the model trains on);
    fall back to an existing path only if you cancel the picker."""
    if IN_COLAB:
        from google.colab import files
        print("  Click 'Choose Files' and select your cohort CSV "
              f"(e.g. {cfg['csv_path']}):")
        up = files.upload()
        if up:
            name = list(up.keys())[0]
            print(f"  Received: {name}  ({len(up[name]):,} bytes)")
            return name
        if os.path.exists(cfg["csv_path"]):
            print(f"  No file chosen; falling back to existing {cfg['csv_path']}")
            return cfg["csv_path"]
        raise RuntimeError("No file uploaded and no local CSV found.")
    return cfg["csv_path"]


# ==============================================================================
# 0.  Configuration
# ==============================================================================
CFG = dict(
    csv_path      = "pediatric_cohort_geraedts_who.csv",
    input_signal  = "observed",     # encoder reads this   ("observed" | "true")
    target_signal = "true",         # model reconstructs this ("true" | "observed")
    coupled       = True,           # False -> three independent univariate models
    compare       = True,           # True -> train coupled AND uncoupled and compare them
    n_epochs      = 300,
    lr            = 1e-3,
    batch_size    = 128,
    ode_method    = "dopri5",       # "dopri5" (adaptive) or "rk4" (fixed, faster)
    launch        = "encoder",      # "encoder": launch state from the encoded history only
                                    # "last_obs": last observed z-score + learned correction
    forecast_w    = 2.5,            # loss weight on forecast horizons vs history
    beta_nll      = 0.5,            # beta-NLL exponent (0 = plain Gaussian NLL)
    seed          = 42,
    outdir        = ".",
    # ---- split -------------------------------------------------------------
    val_ratio     = 0.15,
    test_ratio    = 0.15,           # train_ratio = 1 - val - test = 0.70
    # ---- FIX 3: probabilistic flagging --------------------------------------
    recall_target = 0.80,           # validation recall the flag thresholds aim for
    # ---- FIX 4: faltering augmentation (training set only) ------------------
    augment_frac  = 0.15,           # synthetic faltering children / real training children
    falter_rate   = (0.02, 0.06),   # drift in z-score units per month
    falter_onset  = (3.0, 15.0),    # drift onset age (months), inside the history window
    falter_cap    = 3.0,            # maximum total drift (z-score units)
    falter_up_frac= 0.20,           # share of synthetic children drifting upward
    # ---- illustrative children ----------------------------------------------
    child_id      = 201,            # critical child, forced into TEST
    typical_child = 1008,           # healthy child, forced into TEST ("auto" | a Child_ID | None)
    experiment_tag= "E2_coupled_v2",
    custom_child  = None,           # dict {age: [W,L,C]} over 0-24 m (birth required)
)

# Zone cut-offs in z-space (the p15/p85 and p3/p97 bands):
Z_HEALTHY = 1.0364          # Phi^-1(0.85)
Z_WARNING = 1.8808          # Phi^-1(0.97)
Z90       = 1.6449          # half-width of a 90% Gaussian interval
ZONES     = ["Healthy", "Warning", "Critical"]

ZONE_COLORS = {"Healthy": "#27ae60", "Warning": "#e67e22", "Critical": "#c0392b",
               "Unknown": "#95a5a6"}
STYLE = {
    "bg": "#fafafa", "hist_line": "#2c3e50", "pred_line": "#8e44ad",
    "band_green": "#2ecc71", "band_warn": "#ffb6c1", "band_crit": "#c0392b",
    "grid": "#dfe6e9", "vline": "#636e72",
}


# ==============================================================================
# 1.  Task grid
# ==============================================================================
OBS_AGES      = [0, 1.5, 3, 6, 9, 12, 18, 24]     # historical (encoder) window
FORECAST_AGES = [30, 36, 48, 60]                  # supervised + reported horizons
ALL_AGES      = sorted(set(OBS_AGES + FORECAST_AGES))
TSCALE        = 60.0
LATENT_DIM    = 32
N_VARS        = 3

OBS_IDX   = [ALL_AGES.index(a) for a in OBS_AGES]
FUT_IDX   = [ALL_AGES.index(a) for a in FORECAST_AGES]
LAUNCH    = OBS_AGES[-1]                                   # 24 m
FWD_AGES  = [LAUNCH] + FORECAST_AGES                       # 24 -> 60
BWD_AGES  = [LAUNCH] + OBS_AGES[-2::-1]                    # 24 -> 0
OBS_TNORM = torch.tensor([a / TSCALE for a in OBS_AGES], dtype=torch.float32)
T_FWD     = torch.tensor([a / TSCALE for a in FWD_AGES], dtype=torch.float32)
T_BWD     = torch.tensor([a / TSCALE for a in BWD_AGES], dtype=torch.float32)

TRUE_COLS = ["Weight_kg_true", "Length_cm_true", "HeadCircumference_cm_true"]
OBS_COLS  = ["Weight_kg", "Length_cm", "HeadCircumference_cm"]
Z_COLS    = ["z_W", "z_L", "z_C"]
VLABELS   = ["W", "L", "HC"]


# ==============================================================================
# 2.  FIX 1 -- LMS reference recovered from the cohort; value <-> z-score
# ==============================================================================
def _to_z(x, M, S, L):
    tiny = np.abs(L) < 1e-4
    Ls = np.where(tiny, 1.0, L)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(tiny, np.log(x / M) / S, ((x / M) ** Ls - 1.0) / (Ls * S))


def _from_z(z, M, S, L):
    tiny = np.abs(L) < 1e-4
    Ls = np.where(tiny, 1.0, L)
    base = np.clip(1.0 + Ls * S * z, 1e-6, None)
    return np.where(tiny, M * np.exp(S * z), M * base ** (1.0 / Ls))


class LMS:
    """Reference median M, coefficient of variation S and Box-Cox power L per
    variable, tabulated at the visit ages and interpolated between them."""
    def __init__(self, M, S, L):
        self.M, self.S, self.L = M, S, L                      # each (12, 3)
        ages = np.array(ALL_AGES, float)
        self._f = [PchipInterpolator(ages, a, axis=0) for a in (M, S, L)]

    def at(self, ages):
        ages = np.asarray(ages, float)
        return tuple(f(ages) for f in self._f)                # each (T, 3)

    def to_z(self, x, ages=ALL_AGES):
        return _to_z(x, *self.at(ages))

    def from_z(self, z, ages=ALL_AGES):
        return _from_z(z, *self.at(ages))


def recover_lms(raw_true, z_true):
    """At each age x^L is linear in z:  x^L = M^L + M^L * L * S * z.
    L is searched per age, so a constant power and the age-varying WHO power
    are both handled.  raw_true, z_true : (N, 12, 3) for TRAINING children."""
    def fit(z, x, L):
        slope, icpt = np.polyfit(z, x ** L, 1)
        M, S = icpt ** (1.0 / L), slope / (icpt * L)
        return M, S, np.abs(_from_z(z, M, S, np.asarray(L)) - x).max()

    T = raw_true.shape[1]
    M, S, L = (np.zeros((T, N_VARS)) for _ in range(3))
    worst = 0.0
    for a in range(T):
        for v in range(N_VARS):
            z, x = z_true[:, a, v].astype(float), raw_true[:, a, v].astype(float)
            best = min((minimize_scalar(lambda p: fit(z, x, p)[2], bounds=b, method="bounded")
                        for b in ((-1.5, -1e-3), (1e-3, 2.0))), key=lambda r: r.fun)
            Lv = float(np.round(best.x, 3)) or 1e-3
            M[a, v], S[a, v], resid = fit(z, x, Lv)
            L[a, v] = Lv
            worst = max(worst, resid)
    print(f"  LMS reference recovered from training children "
          f"(largest residual {worst:.4f} in kg/cm)")
    if worst > 0.05:
        warnings.warn("LMS recovery residual is large; check the z-score columns.")
    return LMS(M, S, L)


def zone_idx(z):
    a = np.abs(z)
    return (a > Z_HEALTHY).astype(int) + (a > Z_WARNING).astype(int)


def overall_zone(zones):
    return max(zones, key=lambda z: ZONES.index(z))


# ==============================================================================
# 3.  Model -- Bi-GRU encoder, FIX 2 launch at 24 m, FIX 3 uncertainty head
# ==============================================================================
class TrajectoryEncoder(nn.Module):
    """Bi-GRU over [(z-scores, t_norm, mask) x T_obs] -> child context, plus a
    learned correction to the launch state (zero at initialisation)."""
    def __init__(self, n_vars=N_VARS, hidden_dim=64, latent_dim=LATENT_DIM):
        super().__init__()
        self.gru  = nn.GRU(n_vars + 2, hidden_dim, batch_first=True, bidirectional=True)
        self.proj = nn.Sequential(nn.Linear(hidden_dim * 2, latent_dim), nn.Tanh())
        self.launch = nn.Linear(latent_dim, n_vars)
        nn.init.zeros_(self.launch.weight)
        nn.init.zeros_(self.launch.bias)

    def forward(self, seq):
        _, h = self.gru(seq)
        ctx = self.proj(torch.cat([h[0], h[1]], dim=-1))
        return ctx, self.launch(ctx)


class ConditionedODEFunc(nn.Module):
    """dz/dt = f_theta(z, t, ctx).  With n_vars = 3 the variables are coupled
    through shared weights; with n_vars = 1 the model is univariate."""
    def __init__(self, state_dim=N_VARS, latent_dim=LATENT_DIM, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + 1 + latent_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, state_dim),
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.1)
                nn.init.zeros_(m.bias)
        self._ctx = None

    def set_context(self, ctx):
        self._ctx = ctx

    def forward(self, t, h):
        return self.net(torch.cat([h, t.expand(h.shape[0], 1), self._ctx], dim=1))


class SigmaHead(nn.Module):
    """log sigma(t | ctx) per variable: the predictive standard deviation."""
    def __init__(self, n_vars=N_VARS, latent_dim=LATENT_DIM, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(latent_dim + 1, hidden_dim), nn.SiLU(),
                                 nn.Linear(hidden_dim, n_vars))
        nn.init.constant_(self.net[-1].bias, math.log(0.3))

    def forward(self, ctx, t_norm):
        B, T = ctx.shape[0], t_norm.shape[0]
        x = torch.cat([ctx[:, None, :].expand(B, T, -1),
                       t_norm.view(1, T, 1).expand(B, T, 1)], dim=-1)
        return self.net(x).clamp(-4.0, 1.5)


class LatentGrowthODE(nn.Module):
    def __init__(self, n_vars=N_VARS, hidden_enc=64, hidden_ode=128,
                 latent_dim=LATENT_DIM, method="dopri5", launch="encoder"):
        super().__init__()
        self.encoder  = TrajectoryEncoder(n_vars, hidden_enc, latent_dim)
        self.ode_func = ConditionedODEFunc(n_vars, latent_dim, hidden_ode)
        self.sigma    = SigmaHead(n_vars, latent_dim)
        self.method   = method
        self.launch   = launch
        if launch == "encoder":
            # launch state z(24 m) read off the encoded history alone
            self.state_head = nn.Sequential(nn.Linear(latent_dim, 64), nn.SiLU(),
                                            nn.Linear(64, n_vars))

    def _flow(self, h0, t, rtol=1e-4, atol=1e-5):
        if self.method == "rk4":
            out = odeint(self.ode_func, h0, t, method="rk4",
                         options=dict(step_size=1.0 / 60.0))
        else:
            out = odeint(self.ode_func, h0, t, method=self.method, rtol=rtol, atol=atol)
        return out.permute(1, 0, 2)                       # (B, T, n_vars)

    def predict_grid(self, seq, z_last, t_fwd, t_bwd):
        """Launch at t_fwd[0] = t_bwd[0] (24 m) and integrate forward and
        backward in time.  launch="encoder": the launch state comes from the
        encoded history only (z_last is ignored); launch="last_obs": last
        observed z-score plus a learned correction.
        Returns mean_fwd, mean_bwd, log_sigma_fwd, log_sigma_bwd."""
        ctx, corr = self.encoder(seq)
        h0 = self.state_head(ctx) if self.launch == "encoder" else z_last + corr
        self.ode_func.set_context(ctx)
        return (self._flow(h0, t_fwd), self._flow(h0, t_bwd),
                self.sigma(ctx, t_fwd), self.sigma(ctx, t_bwd))


class IndependentGrowthODE(nn.Module):
    """Three univariate models with no shared information (matched baseline)."""
    def __init__(self, **kw):
        super().__init__()
        self.models = nn.ModuleList([LatentGrowthODE(n_vars=1, **kw) for _ in range(N_VARS)])

    def predict_grid(self, seq, z_last, t_fwd, t_bwd):
        outs = [m.predict_grid(seq[..., [v, N_VARS, N_VARS + 1]], z_last[:, [v]], t_fwd, t_bwd)
                for v, m in enumerate(self.models)]
        return tuple(torch.cat([o[k] for o in outs], dim=-1) for k in range(4))


def build_model(cfg):
    kw = dict(method=cfg["ode_method"], launch=cfg.get("launch", "last_obs"))
    return LatentGrowthODE(**kw) if cfg["coupled"] else IndependentGrowthODE(**kw)


def on_visit_grid(model, seq, z_last, device):
    """Mean and log-sigma on ALL_AGES, shape (B, 12, 3)."""
    mf, mb, lf, lb = model.predict_grid(seq, z_last, T_FWD.to(device), T_BWD.to(device))
    return (torch.cat([mb[:, 1:].flip(1), mf], dim=1),
            torch.cat([lb[:, 1:].flip(1), lf], dim=1))


# ==============================================================================
# 4.  Data pipeline
# ==============================================================================
def load_data(path):
    print(f"\n  Loading: {path}")
    df = pd.read_csv(path) if str(path).lower().endswith((".csv", ".tsv")) \
        else pd.read_excel(path)
    required = set(["Child_ID", "Age_Months", "Observed"] + TRUE_COLS + OBS_COLS + Z_COLS)
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    print(f"  {df.Child_ID.nunique()} children | {len(df)} rows "
          f"| ages {sorted(df.Age_Months.unique())}")
    if "kappa" in df.columns:
        print(f"  coupling kappa = {sorted(df.kappa.unique())}")
    return df


def build_arrays(df):
    """For every child with the full 0-60 m grid:
        z_true (N,12,3)  raw_true (N,12,3)  raw_obs (N,12,3, NaN = missed)
        mask (N,12)  ids (N,)."""
    zs, trues, obss, masks, ids = [], [], [], [], []
    for cid, g in df.groupby("Child_ID"):
        g = g.drop_duplicates("Age_Months").set_index("Age_Months")
        if not all(a in g.index for a in ALL_AGES):
            continue
        zs.append(g.loc[ALL_AGES, Z_COLS].values.astype(np.float32))
        trues.append(g.loc[ALL_AGES, TRUE_COLS].values.astype(np.float32))
        obss.append(g.loc[ALL_AGES, OBS_COLS].values.astype(np.float32))
        masks.append(g.loc[ALL_AGES, "Observed"].astype(float).values.astype(np.float32))
        ids.append(cid)
    print(f"  Complete trajectories (0-60 m): {len(ids)}")
    return dict(z_true=np.stack(zs), raw_true=np.stack(trues), raw_obs=np.stack(obss),
                mask=np.stack(masks), ids=np.array(ids))


def split_indices(ids, cfg):
    """70/15/15 split BY CHILD with the critical child (cfg['child_id']) and the
    healthy child (cfg['typical_child'], when a Child_ID) forced into TEST."""
    N, seed = len(ids), cfg["seed"]
    n_test = int(round(cfg["test_ratio"] * N))
    n_val  = int(round(cfg["val_ratio"] * N))
    perm = list(np.random.default_rng(seed).permutation(N))
    id_to_pos = {int(c): i for i, c in enumerate(ids)}
    forced = []
    if cfg.get("child_id") is not None and int(cfg["child_id"]) in id_to_pos:
        forced = [id_to_pos[int(cfg["child_id"])]]
    rest = [p for p in perm if p not in set(forced)]
    test_idx  = forced + rest[:max(0, n_test - len(forced))]
    remaining = rest[max(0, n_test - len(forced)):]
    # The healthy child must be held out too: swap it into TEST if the random
    # split put it in train/val (no change when it is already in TEST).
    typ = cfg.get("typical_child")
    if isinstance(typ, (int, np.integer)) and int(typ) in id_to_pos:
        tp = id_to_pos[int(typ)]
        if tp not in test_idx and tp in remaining:
            i = remaining.index(tp)
            remaining[i], test_idx[-1] = test_idx[-1], tp
    val_idx, train_idx = remaining[:n_val], remaining[n_val:]
    print(f"\n  Split by child (seed {seed}):  "
          f"train {len(train_idx)} | val {len(val_idx)} | test {len(test_idx)}")
    pinned_ids = [int(c) for c in (cfg.get("child_id"), typ)
                  if isinstance(c, (int, np.integer)) and int(c) in id_to_pos
                  and id_to_pos[int(c)] in test_idx]
    if pinned_ids:
        print(f"  Fixed child(ren) {pinned_ids} placed in TEST (never trained on).")
    return np.array(train_idx), np.array(val_idx), np.array(test_idx)


def make_faltering(z_true, z_obs, mask, n, rng, cfg):
    """FIX 4.  Copy n children and add a sustained drift in z-space:
        drift(t) = sign * rate * max(0, t - onset), capped at falter_cap,
    applied with a per-variable pattern.  Noise and missed visits are kept."""
    src   = rng.integers(0, z_true.shape[0], n)
    ages  = np.array(ALL_AGES, float)
    rate  = rng.uniform(*cfg["falter_rate"], n)
    onset = rng.uniform(*cfg["falter_onset"], n)
    sign  = np.where(rng.random(n) < cfg["falter_up_frac"], 1.0, -1.0)
    patterns = np.array([[1.0, 1.0, 0.3],     # weight + length, head mostly spared
                         [1.0, 0.4, 0.2],     # weight-led
                         [1.0, 1.0, 1.0]])    # all three
    pat = patterns[rng.choice(len(patterns), n, p=[0.5, 0.25, 0.25])]
    ramp = np.clip(ages[None, :] - onset[:, None], 0.0, None) * rate[:, None]
    ramp = np.minimum(ramp, cfg["falter_cap"]) * sign[:, None]            # (n, 12)
    drift = (ramp[:, :, None] * pat[:, None, :]).astype(np.float32)       # (n, 12, 3)
    return z_true[src] + drift, z_obs[src] + drift, mask[src].copy()


def make_pack(z_true, z_obs, mask, ids, lms):
    """Tensors for training plus numpy copies for scoring."""
    N = z_true.shape[0]
    zo = np.nan_to_num(z_obs, nan=0.0).astype(np.float32)
    hist = mask[:, OBS_IDX]
    last = (hist * np.arange(len(OBS_IDX))[None, :]).argmax(axis=1)      # last attended visit
    z_last = zo[np.arange(N), np.array(OBS_IDX)[last]]
    return dict(
        z_true=torch.from_numpy(z_true.astype(np.float32)),
        z_obs=torch.from_numpy(zo),
        mask=torch.from_numpy(mask.astype(np.float32)),
        z_last=torch.from_numpy(z_last),
        ids=np.asarray(ids),
        z_true_np=z_true, z_last_np=z_last,
        raw_true=lms.from_z(z_true), raw_obs=lms.from_z(z_obs),
    )


def make_obs_seq(z_win, mask_win, t_norm):
    """z_win (B,To,3), mask_win (B,To), t_norm (To,) -> (B,To,5)."""
    B, T, _ = z_win.shape
    return torch.cat([z_win, t_norm.view(1, T, 1).expand(B, T, 1),
                      mask_win.view(B, T, 1)], dim=-1)


def encoder_inputs(pack, cfg, device, idx=None):
    sel = slice(None) if idx is None else idx
    if cfg["input_signal"] == "true":
        z, m = pack["z_true"][sel], torch.ones_like(pack["mask"][sel])
        z_last = z[:, OBS_IDX[-1], :]
    else:
        z, m, z_last = pack["z_obs"][sel], pack["mask"][sel], pack["z_last"][sel]
    seq = make_obs_seq(z[:, OBS_IDX, :].to(device), m[:, OBS_IDX].to(device),
                       OBS_TNORM.to(device))
    return seq, z_last.to(device)


# ==============================================================================
# 5.  Training -- masked, forecast-weighted Gaussian negative log-likelihood
# ==============================================================================
def train_model(train, val, cfg, device="cpu"):
    model = build_model(cfg).to(device)
    opt = optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=1e-5)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["n_epochs"], eta_min=1e-5)

    w_age = torch.tensor([cfg["forecast_w"] if a in FORECAST_AGES else 1.0
                          for a in ALL_AGES], dtype=torch.float32, device=device)
    w_age = w_age / w_age.mean()
    target_true, beta = (cfg["target_signal"] == "true"), cfg["beta_nll"]

    def loss_on(pack, idx=None):
        sel = slice(None) if idx is None else idx
        seq, z_last = encoder_inputs(pack, cfg, device, idx)
        if target_true:
            tgt = pack["z_true"][sel].to(device)
            tgt_m = torch.ones_like(pack["mask"][sel]).to(device)
        else:
            tgt = pack["z_obs"][sel].to(device)
            tgt_m = pack["mask"][sel].clone().to(device)
            tgt_m[:, FUT_IDX] = 1.0
        mu, ls = on_visit_grid(model, seq, z_last, device)
        var = torch.exp(2.0 * ls)
        nll = 0.5 * (mu - tgt) ** 2 / var + ls
        if beta > 0:
            nll = nll * var.detach() ** beta
        w = w_age.view(1, -1) * tgt_m
        return (nll.mean(-1) * w).sum() / w.sum().clamp_min(1.0)

    N = train["z_true"].shape[0]
    tr_hist, va_hist, best, best_state = [], [], float("inf"), None
    print(f"\n{'-'*70}\n  {'Coupled' if cfg['coupled'] else 'Independent univariate'} "
          f"Neural ODE in z-space  |  input={cfg['input_signal']} "
          f"target={cfg['target_signal']}  |  solver={cfg['ode_method']}  "
          f"|  epochs={cfg['n_epochs']}  |  launch={cfg.get('launch', 'last_obs')}"
          f"\n{'-'*70}")

    for ep in range(1, cfg["n_epochs"] + 1):
        model.train()
        perm = torch.randperm(N)
        run, nb = 0.0, 0
        for s in range(0, N, cfg["batch_size"]):
            loss = loss_on(train, perm[s:s + cfg["batch_size"]])
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); run += loss.item(); nb += 1
        run /= nb; sch.step()

        model.eval()
        with torch.no_grad():
            vl = loss_on(val).item()
        tr_hist.append(run); va_hist.append(vl)
        if vl < best:
            best, best_state = vl, {k: v.clone() for k, v in model.state_dict().items()}
        if ep % 25 == 0 or ep == 1:
            print(f"  Epoch {ep:4d} | train {run:.5f} | val {vl:.5f} "
                  f"| lr {opt.param_groups[0]['lr']:.2e}")

    model.load_state_dict(best_state)
    print(f"\n  Best val loss: {best:.5f}")
    return model, tr_hist, va_hist


# ==============================================================================
# 6.  Inference
# ==============================================================================
def predict_pack(model, pack, cfg, device="cpu"):
    """Predictive mean and standard deviation in z-space, each (N, 12, 3)."""
    model.eval()
    with torch.no_grad():
        seq, z_last = encoder_inputs(pack, cfg, device)
        mu, ls = on_visit_grid(model, seq, z_last, device)
    return mu.cpu().numpy(), np.exp(ls.cpu().numpy())


def zone_probs(mu, sig):
    """P(not Healthy) and P(Critical) under the Gaussian predictive."""
    p_nh = norm.cdf((-Z_HEALTHY - mu) / sig) + norm.sf((Z_HEALTHY - mu) / sig)
    p_c  = norm.cdf((-Z_WARNING - mu) / sig) + norm.sf((Z_WARNING - mu) / sig)
    return p_nh, p_c


def prob_zone(p_nh, p_c, thr):
    """FIX 3 decision rule: Critical if P(Critical) clears its threshold,
    else Warning if P(not Healthy) clears its threshold, else Healthy."""
    return np.where(p_c >= thr["critical"], 2, np.where(p_nh >= thr["nonhealthy"], 1, 0))


def tune_thresholds(model, val, cfg, device="cpu"):
    """Choose both thresholds on VALIDATION forecasts so that the stated share
    of truly Critical (and truly non-Healthy) cases would be flagged."""
    mu, sig = predict_pack(model, val, cfg, device)
    p_nh, p_c = zone_probs(mu[:, FUT_IDX], sig[:, FUT_IDX])
    tz = zone_idx(val["z_true_np"][:, FUT_IDX])
    q = 1.0 - cfg["recall_target"]
    thr = dict(critical=float(np.quantile(p_c[tz == 2], q)) if (tz == 2).any() else 0.5,
               nonhealthy=float(np.quantile(p_nh[tz >= 1], q)) if (tz >= 1).any() else 0.5)
    print(f"\n  Flag thresholds tuned on validation for {cfg['recall_target']:.0%} recall:  "
          f"P(Critical) >= {thr['critical']:.3f},  P(not Healthy) >= {thr['nonhealthy']:.3f}")
    return thr


def predict_child(model, lms, history, cfg, device="cpu"):
    """Forecast one child from its 0-24 m history ({age: [W,L,C]} in kg/cm).
    Returns the dense trajectory in kg/cm with a 90% forecast interval."""
    if 0 not in history:
        raise ValueError("Birth (age 0) measurement is required.")
    bad = [a for a in history if a not in OBS_AGES]
    if bad:
        raise ValueError(f"Ages {bad} are not on the historical grid {OBS_AGES}.")

    raw = np.full((len(OBS_AGES), N_VARS), np.nan)
    for i, a in enumerate(OBS_AGES):
        if a in history:
            raw[i] = history[a]
    mask = (~np.isnan(raw[:, 0])).astype(np.float32)
    z = np.nan_to_num(lms.to_z(raw, OBS_AGES), nan=0.0).astype(np.float32)
    z_last = z[int(np.nonzero(mask)[0][-1])]
    seq = make_obs_seq(torch.tensor(z[None]), torch.tensor(mask[None]), OBS_TNORM).to(device)

    fwd = np.unique(np.concatenate([np.linspace(LAUNCH, 60.0, 145), FORECAST_AGES]))
    bwd = np.linspace(LAUNCH, 0.0, 97)
    model.eval()
    with torch.no_grad():
        mf, mb, lf, lb = model.predict_grid(
            seq, torch.tensor(z_last[None], device=device),
            torch.tensor(fwd / TSCALE, dtype=torch.float32, device=device),
            torch.tensor(bwd / TSCALE, dtype=torch.float32, device=device))
    mf, mb = mf[0].cpu().numpy(), mb[0].cpu().numpy()
    sf = np.exp(lf[0].cpu().numpy())

    present = [a for a in OBS_AGES if a in history]
    cp = {}
    for a in FORECAST_AGES:
        k = int(np.argmin(np.abs(fwd - a)))
        cp[a] = dict(z=mf[k], sig=sf[k], raw=lms.from_z(mf[k][None], [a])[0])
    return dict(obs_ages=present, obs_raw=np.array([history[a] for a in present], float),
                hist_t=bwd[::-1], hist=lms.from_z(mb[::-1], bwd[::-1]),
                fore_t=fwd, fore=lms.from_z(mf, fwd),
                fore_lo=lms.from_z(mf - Z90 * sf, fwd), fore_hi=lms.from_z(mf + Z90 * sf, fwd),
                checkpoints=cp)


# ==============================================================================
# 7.  Evaluation -- accuracy, zones, calibration
# ==============================================================================
def _rmse(e, axis=None):
    return np.sqrt((e ** 2).mean(axis=axis))


def evaluate(model, pack, lms, thr, cfg, device="cpu", split_name="TEST"):
    mu, sig = predict_pack(model, pack, cfg, device)             # z-space (N,12,3)
    z_true, true = pack["z_true_np"], pack["raw_true"]
    N = mu.shape[0]
    pred = lms.from_z(mu)
    med  = lms.from_z(np.zeros_like(mu))                          # population median

    e_m, e_p = (x[:, FUT_IDX] - true[:, FUT_IDX] for x in (pred, med))
    hist  = _rmse(pred[:, OBS_IDX] - true[:, OBS_IDX], axis=(0, 1))
    floor = np.sqrt(np.nanmean((pack["raw_obs"][:, FUT_IDX] - true[:, FUT_IDX]) ** 2, axis=(0, 1)))

    print(f"\n{'='*74}\n  {split_name}-SET FORECAST ACCURACY (vs TRUE trajectory, "
          f"{N} children)\n{'='*74}")
    print(f"  History fit RMSE (0-24 m)  [W,L,HC]: {hist[0]:.3f}, {hist[1]:.3f}, {hist[2]:.3f}")
    print(f"  Measurement-noise floor    [W,L,HC]: {floor[0]:.3f}, {floor[1]:.3f}, {floor[2]:.3f}")
    print(f"\n  RMSE by horizon             Neural ODE")
    print(f"  {'Horizon':<9}{'W':>8}{'L':>8}{'HC':>8}")
    print(f"  {'-'*33}")
    for h, age in enumerate(FORECAST_AGES):
        a = _rmse(e_m[:, h], axis=0)
        print(f"  {int(age):>3}m{'':<5}{a[0]:>8.3f}{a[1]:>8.3f}{a[2]:>8.3f}")
    rmse, rmse_p = (_rmse(e, axis=(0, 1)) for e in (e_m, e_p))
    mae = np.abs(e_m).mean(axis=(0, 1))
    print(f"  {'-'*33}")
    print(f"  {'RMSE':<9}{rmse[0]:>8.3f}{rmse[1]:>8.3f}{rmse[2]:>8.3f}")
    print(f"  {'MAE':<9}{mae[0]:>8.3f}{mae[1]:>8.3f}{mae[2]:>8.3f}")
    print(f"  {'Median':<9}{rmse_p[0]:>8.3f}{rmse_p[1]:>8.3f}{rmse_p[2]:>8.3f}"
          f"   (population-median RMSE)")
    gain_p = 100 * (1 - rmse / rmse_p)
    print(f"\n  Improvement on median (%) [W,L,HC]: "
          f"{gain_p[0]:+.1f}, {gain_p[1]:+.1f}, {gain_p[2]:+.1f}")
    rmse_all, mae_all = float(_rmse(e_m)), float(np.abs(e_m).mean())
    print(f"\n  >> OVERALL {split_name} (pooled over W,L,HC and 30-60 m):  "
          f"RMSE = {rmse_all:.4f}   MAE = {mae_all:.4f}")

    # ---- FIX 3: zones from the point forecast and from the predictive ----------
    tz = zone_idx(z_true[:, FUT_IDX])                              # (N,4,3)
    p_nh, p_c = zone_probs(mu[:, FUT_IDX], sig[:, FUT_IDX])
    rules = {"ODE point": zone_idx(mu[:, FUT_IDX]),
             "ODE prob.": prob_zone(p_nh, p_c, thr)}
    print(f"\n  Risk-zone performance over 30-60 m (per-variable labels)")
    print(f"  {'rule':<11}{'acc W':>8}{'acc L':>8}{'acc HC':>8}"
          f"{'Crit recall':>13}{'Crit precision':>16}{'flagged Crit':>14}")
    print(f"  {'-'*76}")
    zstats = {}
    for name, pz in rules.items():
        acc = (pz == tz).mean(axis=(0, 1)) * 100
        rec  = 100 * (pz[tz == 2] == 2).mean() if (tz == 2).any() else float("nan")
        prec = 100 * (tz[pz == 2] == 2).mean() if (pz == 2).any() else float("nan")
        flag = 100 * (pz == 2).mean()
        zstats[name] = dict(acc=acc, recall=rec, precision=prec, flagged=flag)
        print(f"  {name:<11}{acc[0]:>7.1f}%{acc[1]:>7.1f}%{acc[2]:>7.1f}%"
              f"{rec:>12.1f}%{prec:>15.1f}%{flag:>13.1f}%")
    print(f"\n  Accuracy by horizon (%), ODE point / ODE prob.")
    for h, age in enumerate(FORECAST_AGES):
        a = (rules["ODE point"][:, h] == tz[:, h]).mean(0) * 100
        b = (rules["ODE prob."][:, h] == tz[:, h]).mean(0) * 100
        print(f"  {int(age):>3}m   W {a[0]:.1f}/{b[0]:.1f}   L {a[1]:.1f}/{b[1]:.1f}   "
              f"HC {a[2]:.1f}/{b[2]:.1f}")
    inside = np.abs(z_true[:, FUT_IDX] - mu[:, FUT_IDX]) <= Z90 * sig[:, FUT_IDX]   # (N,4,3)
    cover = inside.mean(axis=(0, 1))
    print(f"\n  Coverage of the 90% predictive interval [W,L,HC]: "
          f"{cover[0]*100:.1f}%, {cover[1]*100:.1f}%, {cover[2]*100:.1f}%   (target 90%)")
    zres = (z_true[:, FUT_IDX] - mu[:, FUT_IDX]) / sig[:, FUT_IDX]
    nll = 0.5 * np.log(2 * np.pi) + np.log(sig[:, FUT_IDX]) + 0.5 * zres ** 2     # (N,4,3)
    nm = nll.mean(axis=(0, 1))
    print(f"  Mean predictive NLL in z-space [W,L,HC]: {nm[0]:.3f}, {nm[1]:.3f}, {nm[2]:.3f}"
          f"   (lower is better)")

    # ---- coupling preservation at 60 m -----------------------------------------
    Cp, Ct = np.corrcoef(mu[:, -1].T), np.corrcoef(z_true[:, -1].T)
    print(f"\n  Cross-variable coupling at 60 m  (z-score correlations)")
    print(f"  {'pair':<6}{'cohort(true)':>14}{'forecast(mean)':>16}")
    for name, a, b in [("W-L", 0, 1), ("W-C", 0, 2), ("L-C", 1, 2)]:
        print(f"  {name:<6}{Ct[a,b]:>14.3f}{Cp[a,b]:>16.3f}")
    print(f"{'='*74}")

    return dict(mu=mu, sig=sig, pred=pred, true=true, pred_z=mu[:, -1], true_z=z_true[:, -1],
                rmse=rmse, mae=mae, rmse_median=rmse_p,
                rmse_all=rmse_all, mae_all=mae_all, zones=zstats,
                err=e_m, inside=inside, nll=nll, cover=cover, rules=rules, true_zone=tz,
                ids=np.asarray(pack["ids"]), corr_pred=Cp, corr_true=Ct)


# ==============================================================================
# 8.  Plots
# ==============================================================================
def smooth(x, y, n=300):
    x, y = np.asarray(x, float), np.asarray(y, float)
    if len(x) < 4:
        return x, y
    o = np.argsort(x); x, y = x[o], y[o]
    xs = np.linspace(x[0], x[-1], n)
    return xs, make_interp_spline(x, y, k=3)(xs)


def plot_loss(tr, va, path):
    fig, ax = plt.subplots(figsize=(9, 4), facecolor=STYLE["bg"])
    ax.set_facecolor(STYLE["bg"]); ep = np.arange(1, len(tr) + 1)
    xs, ys = smooth(ep, tr); xv, yv = smooth(ep, va)
    ax.plot(xs, ys, color=STYLE["hist_line"], lw=2, label="Train")
    ax.plot(xv, yv, color=STYLE["pred_line"], lw=2, ls="--", label="Validation")
    be = int(np.argmin(va)) + 1
    ax.axvline(be, color=STYLE["pred_line"], lw=1, ls=":", alpha=.5)
    ax.scatter([be], [min(va)], color=STYLE["pred_line"], s=55, zorder=5)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Masked weighted Gaussian NLL (z-space)")
    ax.set_title("Neural ODE - Training & Validation Loss", fontweight="bold")
    ax.legend(framealpha=.6); ax.grid(color=STYLE["grid"], lw=.8)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); plt.savefig(path, dpi=170, facecolor=STYLE["bg"], bbox_inches="tight"); plt.close()
    print(f"  Saved: {path}")


def plot_forecast(pred, lms, thr, path, title_suffix=""):
    """Observed history, backward Neural ODE fit and forward Neural ODE
    forecast, all against reference bands drawn from the cohort's own LMS."""
    ylabels = ["Weight (kg)", "Length / Height (cm)", "Head Circumference (cm)"]
    ages = np.linspace(0.0, 60.0, 241)
    band = {z: lms.from_z(np.full((len(ages), N_VARS), z), ages)
            for z in (-3.0, -Z_WARNING, -Z_HEALTHY, Z_HEALTHY, Z_WARNING, 3.0)}
    fig = plt.figure(figsize=(18, 6), facecolor=STYLE["bg"])
    fig.suptitle(f"Neural ODE launched at {int(LAUNCH)} m  ·  Boys  ·  0-60 m{title_suffix}",
                 fontsize=14, fontweight="bold", y=1.01)
    for d, ylabel in enumerate(ylabels):
        ax = fig.add_subplot(1, 3, d + 1); ax.set_facecolor(STYLE["bg"])
        b = {z: band[z][:, d] for z in band}
        ax.fill_between(ages, b[-Z_HEALTHY], b[Z_HEALTHY], alpha=.16,
                        color=STYLE["band_green"], label="Healthy (p15-p85)", zorder=0)
        ax.fill_between(ages, b[-Z_WARNING], b[-Z_HEALTHY], alpha=.55,
                        color=STYLE["band_warn"], label="Warning", zorder=0)
        ax.fill_between(ages, b[Z_HEALTHY], b[Z_WARNING], alpha=.55,
                        color=STYLE["band_warn"], zorder=0)
        ax.fill_between(ages, b[-3.0], b[-Z_WARNING], alpha=.18,
                        color=STYLE["band_crit"], label="Critical (<p3, >p97)", zorder=0)
        ax.fill_between(ages, b[Z_WARNING], b[3.0], alpha=.18,
                        color=STYLE["band_crit"], zorder=0)
        for z, name in [(-Z_WARNING, "p3"), (-Z_HEALTHY, "p15"),
                        (Z_HEALTHY, "p85"), (Z_WARNING, "p97")]:
            ax.plot(ages, b[z], lw=.7, color="#b2bec3", ls="--", alpha=.7)
            ax.text(60.6, b[z][-1], name, fontsize=6.5, color="#636e72", va="center")
        ax.plot(pred["hist_t"], pred["hist"][:, d], color=STYLE["hist_line"], lw=2.4,
                label=f"Neural ODE fit (0-{int(LAUNCH)}m)", zorder=4)
        ax.plot(pred["fore_t"], pred["fore"][:, d], color=STYLE["pred_line"], lw=2.4,
                ls="--", label=f"Neural ODE forecast ({int(LAUNCH)}-60m)", zorder=4)
        z_obs = lms.to_z(pred["obs_raw"], pred["obs_ages"])
        for age, val, zo in zip(pred["obs_ages"], pred["obs_raw"][:, d], z_obs[:, d]):
            ax.scatter(age, val, color=ZONE_COLORS[ZONES[int(zone_idx(zo))]], s=64,
                       zorder=7, edgecolors="white", lw=1.5)
        if "truth" in pred:
            ta = sorted(pred["truth"])
            ax.scatter(ta, [pred["truth"][a][d] for a in ta], marker="x", color="black",
                       s=46, lw=1.6, zorder=8, label="True (held out)")
        ymin, ymax = b[-3.0].min(), b[3.0].max()
        sp = ymax - ymin; ymin, ymax = ymin - .04 * sp, ymax + .04 * sp
        ax.set_ylim(ymin, ymax)
        for h, age in enumerate(FORECAST_AGES):
            c = pred["checkpoints"][age]
            _, p_c = zone_probs(c["z"][d], c["sig"][d])
            zone = ZONES[int(zone_idx(c["z"][d]))]          # band the point forecast sits in
            flagged = bool(p_c >= thr["critical"])          # separate early-warning flag
            col = ZONE_COLORS[zone]
            ax.scatter([age], [c["raw"][d]], color=col, s=92, zorder=9, marker="s",
                       edgecolors="#7b1010" if flagged else "white", lw=2.4 if flagged else 1.8)
            ax.annotate(f"{int(age)}m · {c['raw'][d]:.1f} · {zone}"
                        f"{' · FLAGGED' if flagged else ''}",
                        xy=(age, c["raw"][d]),
                        xytext=(age - 9.0, c["raw"][d] + (ymax - ymin) * (.07 + .11 * (h % 2))),
                        fontsize=6.5, color="white", fontweight="bold",
                        bbox=dict(boxstyle="round,pad=0.3", fc=col, alpha=.9, lw=1.6,
                                  ec="#7b1010" if flagged else "none"),
                        arrowprops=dict(arrowstyle="->", color=col, lw=1), zorder=10)
        ax.scatter([], [], color="white", marker="s", s=70, edgecolors="#7b1010", lw=2.4,
                   label=f"Flagged: P(Critical) >= {thr['critical']:.0%}")
        ax.axvline(LAUNCH, color=STYLE["vline"], lw=1.1, ls=":", alpha=.8)
        ax.set_xlim(-.5, 63); ax.set_xlabel("Age (months)"); ax.set_ylabel(ylabel)
        ax.set_title(ylabel, fontweight="bold", pad=8)
        ax.legend(fontsize=7, framealpha=.5, loc="upper left")
        ax.grid(color=STYLE["grid"], lw=.7); ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(pad=1.5); plt.savefig(path, dpi=170, facecolor=STYLE["bg"], bbox_inches="tight"); plt.close()
    print(f"  Saved: {path}")


def plot_coupling(ev, path):
    pz, tz = ev["pred_z"], ev["true_z"]
    pairs = [("z_W", "z_L", 0, 1), ("z_W", "z_C", 0, 2), ("z_L", "z_C", 1, 2)]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), facecolor=STYLE["bg"])
    fig.suptitle("Cross-variable coupling at 60 m - forecast mean vs cohort",
                 fontsize=13, fontweight="bold", y=1.02)
    for ax, (xa, ya, a, b) in zip(axes, pairs):
        ax.set_facecolor(STYLE["bg"])
        ax.scatter(tz[:, a], tz[:, b], s=14, alpha=.35, color="#95a5a6",
                   label=f"cohort r={np.corrcoef(tz[:,a],tz[:,b])[0,1]:.2f}")
        ax.scatter(pz[:, a], pz[:, b], s=14, alpha=.5, color=STYLE["pred_line"],
                   label=f"forecast r={np.corrcoef(pz[:,a],pz[:,b])[0,1]:.2f}")
        ax.axhline(0, color="#dfe6e9", lw=.8); ax.axvline(0, color="#dfe6e9", lw=.8)
        ax.set_xlabel(xa); ax.set_ylabel(ya); ax.legend(framealpha=.6, fontsize=9)
        ax.grid(color=STYLE["grid"], lw=.6); ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); plt.savefig(path, dpi=170, facecolor=STYLE["bg"], bbox_inches="tight"); plt.close()
    print(f"  Saved: {path}")


# ==============================================================================
# 9.  Single-child forecast (from CSV or hand-typed) with truth scoring
# ==============================================================================
def get_child_window(df, child_id):
    """A child's OBSERVED 0-24 m window (skipping missed visits) plus its TRUE
    30-60 m values and zone labels for scoring."""
    g = df[df.Child_ID == child_id].drop_duplicates("Age_Months").set_index("Age_Months")
    if len(g) == 0:
        raise ValueError(f"Child_ID {child_id} not found.")
    if 0 not in g.index:
        raise ValueError(f"Child {child_id} has no birth record.")
    history = {}
    for a in OBS_AGES:
        if a in g.index and bool(g.loc[a, "Observed"]) \
                and not np.isnan(g.loc[a, OBS_COLS[0]]):
            history[a] = [float(g.loc[a, c]) for c in OBS_COLS]
    truth = {a: [float(g.loc[a, c]) for c in TRUE_COLS]
             for a in FORECAST_AGES if a in g.index}
    tzone = {a: [ZONES[int(zone_idx(float(g.loc[a, z])))] for z in Z_COLS]
             for a in FORECAST_AGES if a in g.index}
    return history, truth, tzone


def select_typical_child(test, exclude=None):
    """Rule fixed in advance and using the 0-24 m history ONLY: the test child
    whose attended history visits have the smallest mean |z|."""
    z = np.abs(test["z_obs"].numpy()[:, OBS_IDX]).mean(-1)          # (N, 8)
    m = test["mask"].numpy()[:, OBS_IDX]
    score = (z * m).sum(1) / m.sum(1)
    if exclude is not None:
        score[test["ids"] == exclude] = np.inf
    return int(test["ids"][int(np.argmin(score))])


FULL_COLS = ["age", "var", "forecast", "lo90", "hi90", "z_forecast", "sigma_z",
             "p_not_healthy", "p_critical", "point_zone", "prob_zone",
             "truth", "err", "z_truth", "z_err", "std_err", "in_pi90", "true_zone"]


def full_child_stats(pred, lms, thr, truth=None, tzone=None):
    """Per-horizon, per-variable statistics for one child, from the model's
    predictive distribution (mean and sigma in z-space) and, when supplied, the
    TRUE values.  The 90% interval is mean +/- Z90 * sigma in z-space, mapped
    back to kg / cm."""
    rows = []
    for age in FORECAST_AGES:
        c = pred["checkpoints"][age]
        z, sig = c["z"], c["sig"]
        lo = lms.from_z((z - Z90 * sig)[None], [age])[0]
        hi = lms.from_z((z + Z90 * sig)[None], [age])[0]
        p_nh, p_c = zone_probs(z, sig)
        zp = [ZONES[int(i)] for i in zone_idx(z)]
        zf = [ZONES[int(i)] for i in prob_zone(p_nh, p_c, thr)]
        z_true = lms.to_z(np.array([truth[age]], float), [age])[0] \
            if truth and age in truth else None
        for v in range(N_VARS):
            r = dict(age=int(age), var=VLABELS[v], forecast=float(c["raw"][v]),
                     lo90=float(lo[v]), hi90=float(hi[v]), z_forecast=float(z[v]),
                     sigma_z=float(sig[v]), p_not_healthy=float(p_nh[v]),
                     p_critical=float(p_c[v]), point_zone=zp[v], prob_zone=zf[v])
            if z_true is not None:
                tv = float(truth[age][v])
                r.update(truth=tv, err=float(c["raw"][v]) - tv, z_truth=float(z_true[v]),
                         z_err=float(z[v]) - float(z_true[v]),
                         std_err=(float(z[v]) - float(z_true[v])) / float(sig[v]),
                         in_pi90=bool(lo[v] <= tv <= hi[v]),
                         true_zone=tzone[age][v] if tzone and age in tzone else "")
            rows.append(r)
    return rows


def print_full_stats(rows, has_truth):
    print(f"\n  FULL STATISTICS   (90% interval = mean +/- {Z90:.3f} sigma in z-space;"
          f" P(notH) = P(not Healthy), P(Crit) = P(Critical))")
    head = (f"  {'Age':>4} {'Var':<3} | {'Forecast':>8} {'90% interval':>17} | {'z_fc':>6} "
            f"{'sigma':>6} | {'P(notH)':>7} {'P(Crit)':>7} | {'point':<8} {'prob.':<8}")
    if has_truth:
        head += (f" | {'True':>7} {'Error':>7} {'z_true':>6} {'std.err':>7} "
                 f"{'in PI':>5} {'true zone':<9}")
    print(head)
    print("  " + "-" * (len(head) - 2))
    for r in rows:
        interval = "[%.2f, %.2f]" % (r["lo90"], r["hi90"])
        line = (f"  {r['age']:>3}m {r['var']:<3} | {r['forecast']:>8.2f} {interval:>17} | "
                f"{r['z_forecast']:>+6.2f} {r['sigma_z']:>6.2f} | {r['p_not_healthy']:>7.2f} "
                f"{r['p_critical']:>7.2f} | {r['point_zone']:<8} {r['prob_zone']:<8}")
        if has_truth and "truth" in r:
            line += (f" | {r['truth']:>7.2f} {r['err']:>+7.2f} {r['z_truth']:>+6.2f} "
                     f"{r['std_err']:>+7.2f} {('yes' if r['in_pi90'] else 'no'):>5} "
                     f"{r['true_zone']:<9}")
        print(line)

    T = [r for r in rows if "truth" in r]
    if not T:
        return
    print(f"\n  Summary vs TRUE   (RMSE / MAE in kg or cm; 'z' columns in z-score units)")
    print(f"  {'':<7}{'RMSE':>7}{'MAE':>7}{'RMSE z':>8}{'MAE z':>8}{'in 90% PI':>11}"
          f"{'point acc':>11}{'prob. acc':>11}")
    groups = [(VLABELS[v], [r for r in T if r["var"] == VLABELS[v]]) for v in range(N_VARS)]
    for name, g in groups + [("pooled", T)]:
        e = np.array([r["err"] for r in g])
        ez = np.array([r["z_err"] for r in g])
        val = (f"{np.sqrt((e ** 2).mean()):>7.3f}{np.abs(e).mean():>7.3f}"
               if name != "pooled" else f"{'-':>7}{'-':>7}")
        print(f"  {name:<7}{val}{np.sqrt((ez ** 2).mean()):>8.3f}{np.abs(ez).mean():>8.3f}"
              f"{sum(r['in_pi90'] for r in g):>7d}/{len(g):<3d}"
              f"{100 * np.mean([r['point_zone'] == r['true_zone'] for r in g]):>10.1f}%"
              f"{100 * np.mean([r['prob_zone'] == r['true_zone'] for r in g]):>10.1f}%")

    def first(key):
        for age in FORECAST_AGES:
            if any(r[key] == "Critical" for r in T if r["age"] == age):
                return f"{int(age)} m"
        return "never"
    msg = (f"\n  Earliest horizon with a Critical call (any variable):  "
           f"point {first('point_zone')}  |  probability rule {first('prob_zone')}")
    if any(r["true_zone"] for r in T):
        msg += f"  |  truth {first('true_zone')}"
    print(msg)


def assess_child(model, lms, thr, history, truth=None, tzone=None,
                 cfg=CFG, device="cpu", label="", tag="", plot=True, outdir=".",
                 role="", setup=""):
    """Forecast one child (0-24 -> 24-60), GRAPH it, CLASSIFY each horizon by
    the point forecast and by the probability rule, SCORE it against truth and
    print FULL STATISTICS.  Appends tagged rows to child_<label>_experiments.csv
    and the full statistics to child_<label>_full_stats.csv."""
    pred = predict_child(model, lms, history, cfg, device)
    if truth:
        pred["truth"] = truth
    R = dict(label=label, tag=tag, role=role, checkpoints=pred["checkpoints"],
             has_truth=truth is not None)

    print(f"\n{'#'*78}\n  CHILD FORECAST{('  ·  ' + label) if label else ''}"
          f"{('  ['+tag+']') if tag else ''}"
          f"    (encode 0-{int(LAUNCH)} m  ->  forecast {int(LAUNCH)}-60 m)\n{'#'*78}")
    print(f"  History visits supplied: {sorted(history.keys())} m"
          f"{'   [some visits missing]' if len(history) < len(OBS_AGES) else ''}")
    if role or setup:
        print(f"  Role: {role or '-'}   |   Setup: {setup or '-'}")
    if role and tzone and 60 in tzone:
        true60 = overall_zone(tzone[60])
        print(f"  True zone at 60 m: {true60}"
              f"{'' if true60.lower() == role.lower() else '   <-- differs from the declared role'}")
    print(f"\n  {'Age':>5} | {'Weight':>7} {'Length':>7} {'HeadC':>7} | "
          f"{'point zones':<27} | {'P(Critical) W/L/HC':<19} | flagged")
    print(f"  {'-'*100}")
    zrows = []
    for age in FORECAST_AGES:
        c = pred["checkpoints"][age]
        p_nh, p_c = zone_probs(c["z"], c["sig"])
        zp = [ZONES[int(i)] for i in zone_idx(c["z"])]
        zf = [ZONES[int(i)] for i in prob_zone(p_nh, p_c, thr)]
        zrows.append((age, c["raw"], zp, zf, p_c))
        line = (f"  {int(age):>4}m | {c['raw'][0]:>7.2f} {c['raw'][1]:>7.2f} {c['raw'][2]:>7.2f} | "
                f"{'/'.join(zp):<27} | {p_c[0]:.2f} / {p_c[1]:.2f} / {p_c[2]:.2f}{'':<5} | "
                f"{overall_zone(zf)}")
        if tzone and age in tzone:
            line += f"   (true {overall_zone(tzone[age])})"
        print(line)

    if truth:
        ages = [a for a in FORECAST_AGES if a in truth]
        errs = np.stack([np.abs(pred["checkpoints"][a]["raw"] - np.array(truth[a])) for a in ages])
        rmse, mae = np.sqrt((errs ** 2).mean(0)), errs.mean(0)
        R.update(E_rmse=rmse, E_mae=mae, overall_rmse=float(np.sqrt((errs ** 2).mean())),
                 overall_mae=float(errs.mean()))
        print(f"\n  Forecast error vs TRUE  [W,L,HC]")
        print(f"    RMSE : {rmse[0]:.3f}, {rmse[1]:.3f}, {rmse[2]:.3f}   "
              f"|  pooled RMSE = {R['overall_rmse']:.3f}")
        print(f"    MAE  : {mae[0]:.3f}, {mae[1]:.3f}, {mae[2]:.3f}   "
              f"|  pooled MAE  = {R['overall_mae']:.3f}")
        if tzone:
            rows = [r for r in zrows if r[0] in tzone]
            tot = 3 * len(rows)
            R["zone_acc_point"] = sum(r[2][v] == tzone[r[0]][v] for r in rows for v in range(3)) / tot
            R["zone_acc_prob"]  = sum(r[3][v] == tzone[r[0]][v] for r in rows for v in range(3)) / tot
            print(f"    per-variable zone accuracy: point {R['zone_acc_point']*100:.1f}%  "
                  f"|  probability rule {R['zone_acc_prob']*100:.1f}%")

    verdict = overall_zone(zrows[-1][3])
    print(f"\n  >> 60-month outlook (probability rule): {verdict.upper()}"
          f"{'  (consider clinical review)' if verdict != 'Healthy' else ''}")
    full_rows = full_child_stats(pred, lms, thr, truth, tzone)
    print_full_stats(full_rows, has_truth=bool(truth))
    print(f"{'#'*78}")

    if plot:
        fig_path = os.path.join(
            outdir, f"forecast_{label or 'child'}_{tag or 'run'}.png".replace(" ", "_"))
        plot_forecast(pred, lms, thr, fig_path,
                      title_suffix=f"  ·  {label}{(' (' + role + ')') if role else ''}"
                                   f"{('  ['+tag+']') if tag else ''}")
        R["figure"] = fig_path

    log_path = os.path.join(outdir, f"child_{label or 'child'}_experiments.csv")
    rows = []
    for age, raw, zp, zf, p_c in zrows:
        for v in range(3):
            row = dict(experiment=tag, child=label, age_months=int(age),
                       variable=["W", "L", "C"][v], forecast=round(float(raw[v]), 4),
                       pred_zone=zp[v], p_critical=round(float(p_c[v]), 4), flag_zone=zf[v],
                       truth="", abs_err="", true_zone="")
            if truth and age in truth:
                row["truth"] = round(float(truth[age][v]), 4)
                row["abs_err"] = round(abs(float(raw[v]) - float(truth[age][v])), 4)
                row["true_zone"] = tzone[age][v] if tzone and age in tzone else ""
            rows.append(row)
    pd.DataFrame(rows).to_csv(log_path, mode="a" if os.path.exists(log_path) else "w",
                              header=not os.path.exists(log_path), index=False)
    print(f"  Appended {len(rows)} rows to {log_path}  (tag='{tag}')")
    fs_path = os.path.join(outdir, f"child_{label or 'child'}_full_stats.csv")
    fs = pd.DataFrame(full_rows).reindex(columns=FULL_COLS)
    fs.insert(0, "experiment", tag); fs.insert(1, "child", label)
    fs.to_csv(fs_path, mode="a" if os.path.exists(fs_path) else "w",
              header=not os.path.exists(fs_path), index=False)
    print(f"  Appended {len(fs)} rows of full statistics to {fs_path}  (tag='{tag}')")
    R["full_stats"] = full_rows
    R["log"], R["pred"] = log_path, pred
    return R


# ==============================================================================
# 10.  Orchestration
# ==============================================================================
def train_pipeline(cfg, device="cpu", save=True):
    df = load_data(cfg["csv_path"])
    A = build_arrays(df)
    tr, va, te = split_indices(A["ids"], cfg)

    lms = recover_lms(A["raw_true"][tr], A["z_true"][tr])          # FIX 1 (TRAIN only)
    z_obs = lms.to_z(A["raw_obs"].astype(float)).astype(np.float32)
    z_true, mask, ids = A["z_true"], A["mask"], A["ids"]

    # FIX 4: synthetic faltering children added to TRAIN only
    rng = np.random.default_rng(cfg["seed"] + 1)
    zt, zo, mk, tid = z_true[tr], z_obs[tr], mask[tr], ids[tr]
    n_aug = int(round(cfg["augment_frac"] * len(tr)))
    if n_aug > 0:
        a_zt, a_zo, a_mk = make_faltering(zt, zo, mk, n_aug, rng, cfg)
        zt, zo, mk = (np.concatenate(p) for p in ((zt, a_zt), (zo, a_zo), (mk, a_mk)))
        tid = np.concatenate([tid, -np.arange(1, n_aug + 1)])
        print(f"  Faltering augmentation: {n_aug} synthetic children added to TRAIN "
              f"({len(tr)} real + {n_aug} synthetic)")
    train = make_pack(zt, zo, mk, tid, lms)
    val   = make_pack(z_true[va], z_obs[va], mask[va], ids[va], lms)
    test  = make_pack(z_true[te], z_obs[te], mask[te], ids[te], lms)
    s_zt, s_zo, s_mk = make_faltering(z_true[te], z_obs[te], mask[te], len(te),
                                      np.random.default_rng(cfg["seed"] + 2), cfg)
    stress = make_pack(s_zt, s_zo, s_mk, -np.arange(1, len(te) + 1), lms)

    model, tr_hist, va_hist = train_model(train, val, cfg, device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Trainable parameters: {n_params:,}")
    thr = tune_thresholds(model, val, cfg, device)                 # FIX 3

    ev = evaluate(model, test, lms, thr, cfg, device, split_name="TEST")
    ev_stress = evaluate(model, stress, lms, thr, cfg, device,
                         split_name="FALTERING-STRESS")

    op = lambda n: os.path.join(cfg["outdir"], n)
    os.makedirs(cfg["outdir"], exist_ok=True)
    plot_loss(tr_hist, va_hist, op("loss_curve.png"))
    plot_coupling(ev, op("coupling_60m.png"))
    if save:
        torch.save(dict(model_state=model.state_dict(), lms_M=lms.M, lms_S=lms.S, lms_L=lms.L,
                        thresholds=thr, cfg=cfg, obs_ages=OBS_AGES,
                        forecast_ages=FORECAST_AGES), op("coupled_growth_ode_v2.pt"))
        print(f"  Model saved: {op('coupled_growth_ode_v2.pt')}")
    return dict(model=model, lms=lms, thr=thr, df=df, train=train, val=val, test=test,
                stress=stress, ev=ev, ev_stress=ev_stress, cfg=cfg, device=device,
                n_params=n_params)


def forecast_and_assess(ctx, child_id=None, custom_child=None, label=None, tag=None,
                        role=""):
    """Pick a child (by CSV id) or a hand-typed child, then forecast + graph +
    classify + (if CSV) score, tagging the run for cross-experiment use."""
    model, lms, thr, df = ctx["model"], ctx["lms"], ctx["thr"], ctx.get("df")
    cfg, device = ctx["cfg"], ctx.get("device", "cpu")
    tag = tag or cfg.get("experiment_tag", "run")
    kappa = sorted(df.kappa.unique()) if df is not None and "kappa" in df.columns else []
    setup = (("fully coupled" if cfg["coupled"] else "independent (uncoupled)") + " model"
             + (f", cohort kappa = {kappa[0]:g}" if len(kappa) == 1 else ""))
    if custom_child is not None:
        return assess_child(model, lms, thr, custom_child, cfg=cfg, device=device,
                            label=label or "custom", tag=tag, outdir=cfg["outdir"],
                            role=role, setup=setup)
    if child_id is None:
        raise ValueError("Provide child_id (from the CSV) or custom_child (a dict).")
    history, truth, tzone = get_child_window(df, child_id)
    return assess_child(model, lms, thr, history, truth, tzone, cfg=cfg, device=device,
                        label=label or f"child_{child_id}", tag=tag, outdir=cfg["outdir"],
                        role=role, setup=setup)


def assess_children(ctx):
    """Forecast, graph, classify and score the critical and healthy children
    (or the hand-typed child) with the model in ctx."""
    cfg = ctx["cfg"]
    if cfg.get("custom_child"):
        return [forecast_and_assess(ctx, custom_child=cfg["custom_child"], label="custom")]
    results = []
    cid = cfg.get("child_id")
    if cid is None or int(cid) not in set(int(c) for c in ctx["test"]["ids"]):
        cid = int(ctx["test"]["ids"][0])
    results.append(forecast_and_assess(ctx, child_id=cid, role="critical"))
    typ = cfg.get("typical_child")
    if typ == "auto":
        typ = select_typical_child(ctx["test"], exclude=cid)
        print(f"\n  Typical child chosen by rule (smallest mean |z| over 0-24 m "
              f"among test children): Child_ID {typ}")
    if typ is not None:
        results.append(forecast_and_assess(ctx, child_id=int(typ), role="healthy"))
    return results


def show_and_download(fig_paths, outdir):
    """In Colab: show the figures and download everything in outdir as a zip."""
    if not IN_COLAB:
        return
    from IPython.display import Image, display
    for p in fig_paths:
        if p and os.path.exists(p):
            display(Image(p))
    try:
        import shutil
        from google.colab import files
        files.download(shutil.make_archive(os.path.join(outdir, "coupled_ode_outputs"),
                                           "zip", outdir))
    except Exception as e:
        print(f"  (auto-download skipped: {e})")


# ==============================================================================
# 11.  Coupled vs uncoupled comparison (paired over the same test children)
# ==============================================================================
def _paired_ci(fa, fb, n, n_boot=2000, seed=0):
    """95% paired-bootstrap interval for fa(idx) - fb(idx), resampling children."""
    rng = np.random.default_rng(seed)
    d = np.array([np.atleast_1d(np.asarray(fa(i), float) - np.asarray(fb(i), float))
                  for i in (rng.integers(0, n, n) for _ in range(n_boot))])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanpercentile(d, 2.5, axis=0), np.nanpercentile(d, 97.5, axis=0)


def _recall(pz, tz):
    m = tz == 2
    return (pz[m] == 2).mean() if m.any() else np.nan


def _precision(pz, tz):
    m = pz == 2
    return (tz[m] == 2).mean() if m.any() else np.nan


def _verdict(lo, hi, better):
    if not (np.isfinite(lo) and np.isfinite(hi)):
        return "n/a"
    if lo <= 0 <= hi:
        return "no clear difference"
    if better is None:
        return "coupled higher" if lo > 0 else "coupled lower"
    coupled_wins = (hi < 0) if better == "lower" else (lo > 0)
    return "coupled better" if coupled_wins else "uncoupled better"


# (name, one value per variable?, metric(ev, idx), which direction is better)
_P, _Q = "ODE prob.", "ODE point"
COMPARE_METRICS = [
    ("RMSE",                   True,  lambda ev, i: _rmse(ev["err"][i], axis=(0, 1)), "lower"),
    ("MAE",                    True,  lambda ev, i: np.abs(ev["err"][i]).mean(axis=(0, 1)), "lower"),
    ("NLL (z-space)",          True,  lambda ev, i: ev["nll"][i].mean(axis=(0, 1)), "lower"),
    ("90% coverage %",         True,  lambda ev, i: 100 * ev["inside"][i].mean(axis=(0, 1)), None),
    ("Zone acc % (point)",     True,  lambda ev, i: 100 * (ev["rules"][_Q][i] == ev["true_zone"][i]).mean(axis=(0, 1)), "higher"),
    ("Zone acc % (prob.)",     True,  lambda ev, i: 100 * (ev["rules"][_P][i] == ev["true_zone"][i]).mean(axis=(0, 1)), "higher"),
    ("Crit recall % (point)",  False, lambda ev, i: 100 * _recall(ev["rules"][_Q][i], ev["true_zone"][i]), "higher"),
    ("Crit precision % (point)", False, lambda ev, i: 100 * _precision(ev["rules"][_Q][i], ev["true_zone"][i]), "higher"),
    ("Crit recall % (prob.)",  False, lambda ev, i: 100 * _recall(ev["rules"][_P][i], ev["true_zone"][i]), "higher"),
    ("Crit precision % (prob.)", False, lambda ev, i: 100 * _precision(ev["rules"][_P][i], ev["true_zone"][i]), "higher"),
    ("Flagged Crit % (prob.)", False, lambda ev, i: 100 * (ev["rules"][_P][i] == 2).mean(), None),
]


def compare_split(ev_c, ev_u, split_name):
    if not np.array_equal(ev_c["ids"], ev_u["ids"]):
        raise RuntimeError(f"{split_name}: the two runs were scored on different children.")
    n, idx = len(ev_c["ids"]), np.arange(len(ev_c["ids"]))
    print(f"\n{'='*92}\n  COUPLED vs UNCOUPLED  ·  {split_name}  ({n} children, paired)"
          f"\n  diff = coupled - uncoupled;  95% interval from 2000 paired bootstrap "
          f"resamples of children\n{'='*92}")
    print(f"  {'metric':<26}{'var':<5}{'coupled':>9}{'uncoupled':>11}{'diff':>9}"
          f"{'95% interval':>21}   verdict")
    print(f"  {'-'*90}")
    rows = []
    for name, per_var, f, better in COMPARE_METRICS:
        vc, vu = np.atleast_1d(f(ev_c, idx)), np.atleast_1d(f(ev_u, idx))
        lo, hi = _paired_ci(lambda i: f(ev_c, i), lambda i: f(ev_u, i), n)
        for k, lab in enumerate(VLABELS if per_var else ["all"]):
            v = _verdict(lo[k], hi[k], better)
            print(f"  {name:<26}{lab:<5}{vc[k]:>9.3f}{vu[k]:>11.3f}{vc[k] - vu[k]:>+9.3f}"
                  f"{'[%+.3f, %+.3f]' % (lo[k], hi[k]):>21}   {v}")
            rows.append(dict(split=split_name, metric=name, variable=lab,
                             coupled=float(vc[k]), uncoupled=float(vu[k]),
                             diff=float(vc[k] - vu[k]), ci_lo=float(lo[k]),
                             ci_hi=float(hi[k]), verdict=v))
        if name in ("NLL (z-space)", "90% coverage %", "Zone acc % (prob.)",
                    "Crit precision % (point)"):
            print(f"  {'-'*90}")

    print(f"\n  RMSE by horizon\n  {'':<9}{'coupled':^24}  |{'uncoupled':^24}")
    print(f"  {'Horizon':<9}{'W':>8}{'L':>8}{'HC':>8}  |{'W':>8}{'L':>8}{'HC':>8}")
    for h, age in enumerate(FORECAST_AGES):
        a, b = _rmse(ev_c["err"][:, h], axis=0), _rmse(ev_u["err"][:, h], axis=0)
        print(f"  {int(age):>3}m{'':<5}{a[0]:>8.3f}{a[1]:>8.3f}{a[2]:>8.3f}  |"
              f"{b[0]:>8.3f}{b[1]:>8.3f}{b[2]:>8.3f}")
        for v in range(N_VARS):
            rows.append(dict(split=split_name, metric=f"RMSE at {int(age)} m", variable=VLABELS[v],
                             coupled=float(a[v]), uncoupled=float(b[v]), diff=float(a[v] - b[v]),
                             ci_lo=np.nan, ci_hi=np.nan, verdict=""))

    print(f"\n  Cross-variable coupling at 60 m (z-score correlations)")
    print(f"  {'pair':<6}{'cohort(true)':>14}{'coupled':>10}{'uncoupled':>11}")
    for name, a, b in [("W-L", 0, 1), ("W-C", 0, 2), ("L-C", 1, 2)]:
        t, c, u = ev_c["corr_true"][a, b], ev_c["corr_pred"][a, b], ev_u["corr_pred"][a, b]
        print(f"  {name:<6}{t:>14.3f}{c:>10.3f}{u:>11.3f}")
        rows.append(dict(split=split_name, metric="corr at 60 m (cohort true = "
                         f"{t:.3f})", variable=name, coupled=float(c), uncoupled=float(u),
                         diff=float(c - u), ci_lo=np.nan, ci_hi=np.nan, verdict=""))
    return rows


def _first_critical(rows, key):
    for age in FORECAST_AGES:
        if any(r.get(key) == "Critical" for r in rows if r["age"] == age):
            return f"{int(age)} m"
    return "never"


def compare_children(ch_c, ch_u):
    by_u = {R["label"]: R for R in ch_u}
    out = []
    for Rc in ch_c:
        Ru = by_u.get(Rc["label"])
        if Ru is None or "full_stats" not in Rc or "full_stats" not in Ru:
            continue
        rc_rows, ru_rows = Rc["full_stats"], Ru["full_stats"]
        assert [(r["age"], r["var"]) for r in rc_rows] == [(r["age"], r["var"]) for r in ru_rows]
        lab, role = Rc["label"], Rc.get("role", "")
        has_t = all("truth" in r for r in rc_rows)
        print(f"\n{'#'*92}\n  CHILD COMPARISON  ·  {lab}{(' (' + role + ')') if role else ''}"
              f"  ·  coupled (C) vs uncoupled (U)\n{'#'*92}")
        head = f"  {'Age':>4} {'Var':<3} |"
        if has_t:
            head += f" {'True':>7} {'true zone':<9} |"
        head += (f" {'C fcst':>7} {'C err':>6} {'C P(Cr)':>7} {'C prob.':<8} |"
                 f" {'U fcst':>7} {'U err':>6} {'U P(Cr)':>7} {'U prob.':<8}")
        print(head)
        print("  " + "-" * (len(head) - 2))
        for rc, ru in zip(rc_rows, ru_rows):
            ec = f"{rc['err']:>+6.2f}" if has_t else f"{'-':>6}"
            eu = f"{ru['err']:>+6.2f}" if has_t else f"{'-':>6}"
            line = f"  {rc['age']:>3}m {rc['var']:<3} |"
            if has_t:
                line += f" {rc['truth']:>7.2f} {rc['true_zone']:<9} |"
            line += (f" {rc['forecast']:>7.2f} {ec} {rc['p_critical']:>7.2f} {rc['prob_zone']:<8} |"
                     f" {ru['forecast']:>7.2f} {eu} {ru['p_critical']:>7.2f} {ru['prob_zone']:<8}")
            print(line)
            row = dict(child=lab, role=role, age=rc["age"], var=rc["var"],
                       truth=rc.get("truth"), true_zone=rc.get("true_zone"))
            for tagp, r in (("coupled", rc), ("uncoupled", ru)):
                row.update({f"{tagp}_{k}": r.get(k) for k in
                            ("forecast", "lo90", "hi90", "err", "z_forecast", "sigma_z",
                             "p_not_healthy", "p_critical", "point_zone", "prob_zone",
                             "in_pi90", "std_err")})
            out.append(row)
        if not has_t:
            continue
        print(f"\n  {'':<7}{'RMSE C':>8}{'RMSE U':>8}{'in PI C':>9}{'in PI U':>9}"
              f"{'point C':>9}{'point U':>9}{'prob. C':>9}{'prob. U':>9}")
        for v in VLABELS + ["pooled"]:
            gc = [r for r in rc_rows if v == "pooled" or r["var"] == v]
            gu = [r for r in ru_rows if v == "pooled" or r["var"] == v]
            rm = (lambda g: f"{np.sqrt(np.mean([r['err'] ** 2 for r in g])):>8.3f}") \
                if v != "pooled" else (lambda g: f"{'-':>8}")
            pi = lambda g: f"{sum(r['in_pi90'] for r in g)}/{len(g)}"
            acc = lambda g, k: 100 * np.mean([r[k] == r["true_zone"] for r in g])
            print(f"  {v:<7}{rm(gc)}{rm(gu)}{pi(gc):>9}{pi(gu):>9}"
                  f"{acc(gc, 'point_zone'):>8.1f}%{acc(gu, 'point_zone'):>8.1f}%"
                  f"{acc(gc, 'prob_zone'):>8.1f}%{acc(gu, 'prob_zone'):>8.1f}%")
        print(f"\n  Earliest Critical call (any variable), probability rule:  "
              f"coupled {_first_critical(rc_rows, 'prob_zone')}  |  "
              f"uncoupled {_first_critical(ru_rows, 'prob_zone')}  |  "
              f"truth {_first_critical(rc_rows, 'true_zone')}")
    return out


def compare_runs(runs, outdir):
    c, u = runs["coupled"]["ctx"], runs["uncoupled"]["ctx"]
    print(f"\n{'='*92}\n  EXPERIMENT 1 vs 2  ·  COUPLED vs UNCOUPLED NEURAL ODE"
          f"\n  same data, split, augmentation, stress set, seed, epochs and loss\n{'='*92}")
    print(f"  Trainable parameters:  coupled {c['n_params']:,}   |   uncoupled {u['n_params']:,}"
          f"   (uncoupled / coupled = {u['n_params'] / c['n_params']:.2f})")
    print(f"  Flag thresholds (tuned on validation per model):  "
          f"coupled P(Crit) >= {c['thr']['critical']:.3f}, P(notH) >= {c['thr']['nonhealthy']:.3f}"
          f"   |   uncoupled P(Crit) >= {u['thr']['critical']:.3f}, "
          f"P(notH) >= {u['thr']['nonhealthy']:.3f}")
    rows = compare_split(c["ev"], u["ev"], "TEST")
    rows += compare_split(c["ev_stress"], u["ev_stress"], "FALTERING-STRESS")
    rows.append(dict(split="model", metric="trainable parameters", variable="all",
                     coupled=c["n_params"], uncoupled=u["n_params"],
                     diff=c["n_params"] - u["n_params"], ci_lo=np.nan, ci_hi=np.nan, verdict=""))
    child_rows = compare_children(runs["coupled"]["children"], runs["uncoupled"]["children"])
    os.makedirs(outdir, exist_ok=True)
    p1 = os.path.join(outdir, "comparison_summary.csv")
    p2 = os.path.join(outdir, "comparison_children.csv")
    pd.DataFrame(rows).to_csv(p1, index=False)
    pd.DataFrame(child_rows).to_csv(p2, index=False)
    print(f"\n  Saved: {p1}\n  Saved: {p2}")


def main(cfg):
    torch.manual_seed(cfg["seed"]); np.random.seed(cfg["seed"])
    device = "cuda" if torch.cuda.is_available() else "cpu"
    os.makedirs(cfg["outdir"], exist_ok=True)
    print("\n" + "=" * 74)
    print("  PEDIATRIC GROWTH - NEURAL ODE IN Z-SPACE, LAUNCHED AT 24 MONTHS")
    print("  train / validate / TEST  ->  forecast + graph + classify")
    if cfg.get("compare"):
        print("  mode: coupled vs uncoupled comparison (two models, same data)")
    print("=" * 74 + f"\n  Device: {device}")

    cfg["csv_path"] = resolve_csv(cfg)

    if cfg.get("compare"):
        runs = {}
        ver = "v3" if cfg.get("launch", "last_obs") == "encoder" else "v2"
        for name, coupled, tag in (("coupled", True, f"E2_coupled_{ver}"),
                                   ("uncoupled", False, f"E1_uncoupled_{ver}")):
            rc = dict(cfg, coupled=coupled, experiment_tag=tag,
                      outdir=os.path.join(cfg["outdir"], tag))
            print(f"\n{'='*74}\n  RUN: {name.upper()} MODEL   ->   outputs in {rc['outdir']}"
                  f"\n{'='*74}")
            torch.manual_seed(cfg["seed"]); np.random.seed(cfg["seed"])   # same start for both
            ctx = train_pipeline(rc, device=device, save=True)
            runs[name] = dict(ctx=ctx, children=assess_children(ctx))
        compare_runs(runs, cfg["outdir"])
        figs = [os.path.join(r["ctx"]["cfg"]["outdir"], n) for r in runs.values()
                for n in ("loss_curve.png", "coupling_60m.png")]
        figs += [R.get("figure", "") for r in runs.values() for R in r["children"]]
        show_and_download(figs, cfg["outdir"])
        print()
        for name, r in runs.items():
            print(f"  Done ({name}). TEST overall RMSE = {r['ctx']['ev']['rmse_all']:.4f} | "
                  f"MAE = {r['ctx']['ev']['mae_all']:.4f}")
        print()
        return runs

    ctx = train_pipeline(cfg, device=device, save=True)
    results = assess_children(ctx)
    figs = [os.path.join(cfg["outdir"], n) for n in ("loss_curve.png", "coupling_60m.png")]
    figs += [R.get("figure", "") for R in results]
    show_and_download(figs, cfg["outdir"])
    print(f"\n  Done. TEST overall RMSE = {ctx['ev']['rmse_all']:.4f} | "
          f"MAE = {ctx['ev']['mae_all']:.4f}\n")
    return ctx, results


if __name__ == "__main__":
    if IN_COLAB:
        # Colab injects "-f kernel.json"; skip argparse. Edit CFG above to tune.
        main(CFG)
    else:
        ap = argparse.ArgumentParser()
        ap.add_argument("--csv", default=CFG["csv_path"])
        ap.add_argument("--epochs", type=int, default=CFG["n_epochs"])
        ap.add_argument("--input", choices=["observed", "true"], default=CFG["input_signal"])
        ap.add_argument("--target", choices=["true", "observed"], default=CFG["target_signal"])
        ap.add_argument("--solver", choices=["dopri5", "rk4"], default=CFG["ode_method"])
        ap.add_argument("--uncoupled", action="store_true",
                        help="three independent univariate models (Experiment 1)")
        ap.add_argument("--augment", type=float, default=CFG["augment_frac"],
                        help="synthetic faltering children as a fraction of TRAIN (0 = off)")
        ap.add_argument("--child", type=int, default=CFG["child_id"])
        ap.add_argument("--single", action="store_true",
                        help="train one model only (coupled, or uncoupled with --uncoupled)")
        ap.add_argument("--tag", default=None)
        ap.add_argument("--outdir", default=CFG["outdir"])
        a = ap.parse_args()
        CFG.update(csv_path=a.csv, n_epochs=a.epochs, input_signal=a.input,
                   target_signal=a.target, ode_method=a.solver, coupled=not a.uncoupled,
                   augment_frac=a.augment, child_id=a.child, outdir=a.outdir,
                   compare=CFG["compare"] and not (a.single or a.uncoupled),
                   experiment_tag=a.tag or ("E1_uncoupled_v2" if a.uncoupled else "E2_coupled_v2"))
        main(CFG)

SyntaxError: unterminated string literal (detected at line 725) (1538268102.py, line 725)